# 03 — Monte Carlo Policy Evaluation

Notebook 02 solved the MDP exactly, but it cheated: every backup read the true
transition probabilities out of $P$. Real problems seldom hand you that matrix.
You get a system you can *run*, not a system you can *read*.

So this notebook throws the model away. From here on the only access to the
environment is `step()` — take an action, observe what happens. No $P$, no $R$.

**Monte Carlo policy evaluation** is the most direct thing you can do with that
access. The value of a state is defined as an expected return:

$$v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$$

An expectation is an average. So: run episodes, record the returns that actually
followed each state, and average them. No bootstrapping, no model — just the
definition, applied literally.

We keep $P$ around for one purpose only: **grading**. Notebook 02's exact `v*`
is the answer key, and the whole point of this notebook is watching sampled
estimates walk toward it.

In [ ]:
# --- environment from notebook 01, repeated so this notebook stands alone ---
from __future__ import annotations

from enum import IntEnum
from typing import NamedTuple

import numpy as np

class State(IntEnum):
    NO_INFO = 0           # nothing checked yet
    SALINITY_CHECKED = 1  # feed salinity known
    FOULING_CHECKED = 2   # fouling indicators known
    BOTH_CHECKED = 3      # both kinds of evidence in hand
    SUCCESS = 4           # problem solved (terminal)
    FAILURE = 5           # wrong or unsafe fix submitted (terminal)


class Action(IntEnum):
    CHECK_SALINITY = 0
    CHECK_FOULING = 1
    RUN_SIMULATION = 2
    SUBMIT_DIRECTLY = 3


N_STATES, N_ACTIONS = len(State), len(Action)
TERMINAL_STATES = frozenset({State.SUCCESS, State.FAILURE})
NONTERMINAL = [s for s in State if s not in TERMINAL_STATES]


def is_terminal(state) -> bool:
    return State(state) in TERMINAL_STATES


COST_CHECK = -0.5          # first look at a piece of evidence
COST_REPEAT_CHECK = -1.0   # re-checking something already known: pure waste
REWARD_SIM_SUCCESS = 9.0   # +10 outcome, minus the -1 implicit cost of simulating
REWARD_SIM_FAILURE = -11.0
REWARD_SUBMIT_SUCCESS = 10.0
REWARD_SUBMIT_FAILURE = -10.0

SIM_SUCCESS_PROB = {
    State.NO_INFO: 0.15,
    State.SALINITY_CHECKED: 0.55,
    State.FOULING_CHECKED: 0.45,
    State.BOTH_CHECKED: 0.95,
}
SUBMIT_SUCCESS_PROB = {
    State.NO_INFO: 0.05,
    State.SALINITY_CHECKED: 0.35,
    State.FOULING_CHECKED: 0.25,
    State.BOTH_CHECKED: 0.75,
}


class Transition(NamedTuple):
    prob: float
    next_state: State
    reward: float


# Evidence held in each non-terminal state, used to work out where a check lands.
_EVIDENCE = {
    State.NO_INFO: frozenset(),
    State.SALINITY_CHECKED: frozenset({Action.CHECK_SALINITY}),
    State.FOULING_CHECKED: frozenset({Action.CHECK_FOULING}),
    State.BOTH_CHECKED: frozenset({Action.CHECK_SALINITY, Action.CHECK_FOULING}),
}
_STATE_BY_EVIDENCE = {ev: st for st, ev in _EVIDENCE.items()}


def transitions(state, action) -> tuple[Transition, ...]:
    # Every outcome of taking `action` in `state`, probabilities summing to 1.
    state, action = State(state), Action(action)

    if is_terminal(state):
        return (Transition(1.0, state, 0.0),)

    if action in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
        already_known = action in _EVIDENCE[state]
        next_state = (
            state if already_known
            else _STATE_BY_EVIDENCE[_EVIDENCE[state] | {action}]
        )
        reward = COST_REPEAT_CHECK if already_known else COST_CHECK
        return (Transition(1.0, next_state, reward),)

    if action is Action.RUN_SIMULATION:
        p = SIM_SUCCESS_PROB[state]
        return (
            Transition(p, State.SUCCESS, REWARD_SIM_SUCCESS),
            Transition(1.0 - p, State.FAILURE, REWARD_SIM_FAILURE),
        )

    p = SUBMIT_SUCCESS_PROB[state]
    return (
        Transition(p, State.SUCCESS, REWARD_SUBMIT_SUCCESS),
        Transition(1.0 - p, State.FAILURE, REWARD_SUBMIT_FAILURE),
    )

def transition_tables() -> tuple[np.ndarray, np.ndarray]:
    # Dense tables for exact methods: P[s, a, s'] and expected R[s, a].
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        for a in Action:
            for prob, next_state, reward in transitions(s, a):
                P[s, a, next_state] += prob
                R[s, a] += prob * reward
    return P, R


P, R = transition_tables()

def policy_matrices(pi, P, R):
    # Collapse an MDP + policy into an MRP: (P_pi [S,S], R_pi [S]).
    P_pi = np.einsum("sa,sat->st", pi, P)
    R_pi = np.einsum("sa,sa->s", pi, R)
    return P_pi, R_pi


def deterministic(action_of_state) -> np.ndarray:
    # Build a [S, A] policy matrix from a {state: action} mapping.
    pi = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        pi[s, action_of_state(s)] = 1.0
    return pi


def step(state, action, rng) -> tuple[State, float, bool]:
    # Sample one environment step: (next_state, reward, done).
    # Inverse-CDF sampling from a single uniform draw. The obvious
    # rng.choice(..., p=probs) is ~4x slower per call, which matters once the
    # later notebooks run hundreds of thousands of episodes.
    outcomes = transitions(state, action)
    if len(outcomes) == 1:
        only = outcomes[0]
        return only.next_state, only.reward, is_terminal(only.next_state)
    u, cumulative = rng.random(), 0.0
    for prob, next_state, reward in outcomes:
        cumulative += prob
        if u < cumulative:
            return next_state, reward, is_terminal(next_state)
    last = outcomes[-1]  # float-rounding fallback
    return last.next_state, last.reward, is_terminal(last.next_state)


def rollout(policy_fn, rng, max_steps=20):
    # Run one episode; return a list of (state, action, reward) triples.
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = policy_fn(s, rng)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r))
        s = ns
        if done:
            break
    return traj


def show(traj):
    total = sum(r for _, _, r in traj)
    for s, a, r in traj:
        print(f"  {State(s).name:<18} --{Action(a).name:<16}--> {r:+.1f}")
    print(f"  total (undiscounted) = {total:+.1f}")
    return total

GAMMA = 0.95

# The optimal policy from notebook 02 - this is the policy we will evaluate.
optimal_pi = deterministic(lambda s: {
    State.NO_INFO: Action.CHECK_SALINITY,
    State.SALINITY_CHECKED: Action.CHECK_FOULING,
    State.FOULING_CHECKED: Action.CHECK_SALINITY,
    State.BOTH_CHECKED: Action.RUN_SIMULATION,
}.get(s, Action.RUN_SIMULATION))


def evaluate_exact(pi, gamma=GAMMA):
    P_pi, R_pi = policy_matrices(pi, P, R)
    return np.linalg.solve(np.eye(N_STATES) - gamma * P_pi, R_pi)


V_TRUE = evaluate_exact(optimal_pi)   # the answer key, from notebook 02

print("ground truth v_pi (exact, model-based):")
for s in NONTERMINAL:
    print(f"  {s.name:<18}{V_TRUE[s]:>8.4f}")

## Returns from a sampled episode

First, the mechanics. Given one episode
$S_0, A_0, R_1, S_1, A_1, R_2, \ldots, S_{T-1}, A_{T-1}, R_T$,
the return at each step is the discounted sum of everything that followed:

$$G_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{T-t-1} R_T$$

Computing these efficiently is a **backwards scan**. Working right-to-left,
each return is the current reward plus the discounted return of the next step:

$$G_t = R_{t+1} + \gamma\, G_{t+1}, \qquad G_T = 0$$

One pass, no nested loops.

In [ ]:
def sample_policy(pi):
    # Turn a [S, A] policy matrix into a callable for rollout().
    # Inverse-CDF sampling from one uniform draw: much faster than rng.choice
    # per step, which matters once we run hundreds of thousands of episodes.
    cdf = np.cumsum(pi, axis=1)

    def act(state, rng):
        return Action(int(np.searchsorted(cdf[state], rng.random())))
    return act


def returns_along(traj, gamma=GAMMA):
    # Backwards scan: G_t = R_{t+1} + gamma * G_{t+1}. Returns a list aligned with traj.
    G, out = 0.0, []
    for _, _, r in reversed(traj):
        G = r + gamma * G
        out.append(G)
    return list(reversed(out))


rng = np.random.default_rng(0)
traj = rollout(sample_policy(optimal_pi), rng)

print(f"{'t':>3}  {'state':<18}{'action':<17}{'R_t+1':>8}{'G_t':>9}")
for t, ((s, a, r), G) in enumerate(zip(traj, returns_along(traj))):
    print(f"{t:>3}  {State(s).name:<18}{Action(a).name:<17}{r:>8.1f}{G:>9.4f}")

Read the `G_t` column bottom-up. The last step's return is just its own reward.
Each step above it adds its own reward to $\gamma$ times the one below. The
return at $t{=}0$ is what this episode contributes as a sample of
$v_\pi(\texttt{NO\_INFO})$ — and note it is a **single draw from a random
variable**, not an estimate of anything on its own.

Run the next cell a few times to see how much that single draw moves around.

In [ ]:
rng = np.random.default_rng(1)
draws = [returns_along(rollout(sample_policy(optimal_pi), rng))[0] for _ in range(12)]
print("twelve samples of G_0:", " ".join(f"{g:+.2f}" for g in draws))
print(f"\ntrue v_pi(NO_INFO) = {V_TRUE[State.NO_INFO]:.4f}")
print(f"mean of these 12   = {np.mean(draws):.4f}")

Most episodes return about `+7.2`; occasionally one returns about `-12`. That
bimodality is the 95/5 gamble at `BOTH_CHECKED` showing through. The true value
`6.2454` is a number the process **never actually returns** — it is the average
of two outcomes, not either of them.

This is the central fact about Monte Carlo: individual returns are high-variance
and often unlike the mean, so convergence comes from averaging many of them.

## First-visit vs every-visit

A subtlety: a state can appear more than once in an episode. Two conventions:

- **First-visit MC** — average returns following only the *first* occurrence of
  $s$ in each episode. The samples are then independent across episodes, and the
  estimator is unbiased.
- **Every-visit MC** — average returns following *every* occurrence. Samples
  within an episode are correlated, making it biased for finite data, though it
  is consistent and converges to the same limit.

Both are standard; first-visit is the easier one to reason about.

In [ ]:
def mc_evaluate(pi, n_episodes, rng, gamma=GAMMA, first_visit=True):
    # Monte Carlo policy evaluation. Returns (V, counts, history_of_V).
    total = np.zeros(N_STATES)
    count = np.zeros(N_STATES)
    history = np.zeros((n_episodes, N_STATES))
    act = sample_policy(pi)

    for ep in range(n_episodes):
        traj = rollout(act, rng)
        Gs = returns_along(traj, gamma)
        seen = set()
        for t, ((s, _, _), G) in enumerate(zip(traj, Gs)):
            if first_visit and s in seen:
                continue
            seen.add(s)
            total[s] += G
            count[s] += 1
        history[ep] = np.divide(total, count, out=np.zeros(N_STATES), where=count > 0)

    V = np.divide(total, count, out=np.zeros(N_STATES), where=count > 0)
    return V, count, history


rng = np.random.default_rng(42)
V_mc, counts, hist = mc_evaluate(optimal_pi, 20_000, rng)

print(f"{'state':<18}{'MC estimate':>13}{'exact':>10}{'error':>10}{'visits':>9}")
for s in NONTERMINAL:
    print(f"{s.name:<18}{V_mc[s]:>13.4f}{V_TRUE[s]:>10.4f}"
          f"{V_mc[s] - V_TRUE[s]:>+10.4f}{int(counts[s]):>9}")

After 20,000 episodes the estimates land within a few hundredths of the truth —
with **no knowledge of the dynamics whatsoever**.

One detail deserves a second look: all three errors are *negative*. First-visit
MC is supposed to be unbiased, so is something wrong? The honest way to answer
is to re-run the whole estimate many times with different seeds and check
whether the errors average to zero.

In [ ]:
# Is that all-negative error a bias, or just one unlucky seed?
errs = []
for seed in range(30):
    rng_b = np.random.default_rng(500 + seed)
    V_b, _, _ = mc_evaluate(optimal_pi, 2_000, rng_b)
    errs.append(V_b[State.NO_INFO] - V_TRUE[State.NO_INFO])

errs = np.array(errs)
print(f"30 independent runs of 2,000 episodes, error at NO_INFO:")
print(f"  mean error {errs.mean():+.4f}   (expect ~0 if unbiased)")
print(f"  sd         {errs.std():.4f}")
print(f"  negative in {(errs < 0).sum()}/30 runs   (expect ~15/30)")

The mean error sits near zero and the sign is split roughly evenly — the
estimator is unbiased, and the three negative errors above were one correlated
draw, not a defect. They are correlated because a single run shares its episodes
across all states: an unlucky batch with too many `FAILURE` outcomes pushes
*every* state's estimate down together.

This is a habit worth keeping: **one run tells you almost nothing about an
estimator.** Judging a stochastic method by a single seed is how people conclude
that correct implementations are broken (and, more dangerously, that broken ones
work).

Now look at the visit counts. `NO_INFO` and `SALINITY_CHECKED` are visited
every episode; `FOULING_CHECKED` is visited **zero times**. The optimal policy
always checks salinity first, so that state is simply never reached.

This is a real limitation, not a bug. **Monte Carlo can only evaluate states its
policy actually visits.** Its estimate for an unvisited state is undefined — the
`0.0` printed there is an artifact of the `where=count > 0` guard, not a value.
Dynamic programming had no such problem, because it swept all states whether or
not any policy would go there. This gap is precisely why *exploration* becomes a
first-class concern in model-free RL.

In [ ]:
# Convergence: error against the answer key as episodes accumulate.
checkpoints = [10, 50, 100, 500, 1_000, 5_000, 10_000, 20_000]
print(f"{'episodes':>9}" + "".join(f"{s.name[:9]:>11}" for s in NONTERMINAL if counts[s] > 0)
      + f"{'max |err|':>12}")
for n in checkpoints:
    row = hist[n - 1]
    visited = [s for s in NONTERMINAL if counts[s] > 0]
    err = max(abs(row[s] - V_TRUE[s]) for s in visited)
    print(f"{n:>9}" + "".join(f"{row[s]:>11.4f}" for s in visited) + f"{err:>12.4f}")

## The $1/\sqrt{N}$ law

Monte Carlo error shrinks like $1/\sqrt{N}$: the standard error of a mean over
$N$ samples is $\sigma/\sqrt{N}$. That is a harsh rate — **each extra decimal
digit of accuracy costs 100x more episodes.**

Let us measure it rather than take it on faith. Run many independent MC
estimates at each sample size and look at the spread of the results.

In [ ]:
rng = np.random.default_rng(7)
print(f"{'episodes':>9}{'RMSE':>10}{'ratio':>8}   (expect ~0.707 per 2x data)")
prev = None
for n in [125, 250, 500, 1_000, 2_000, 4_000]:
    sq = []
    for _ in range(40):                     # 40 independent runs at this size
        V_n, _, _ = mc_evaluate(optimal_pi, n, rng)
        sq.append((V_n[State.NO_INFO] - V_TRUE[State.NO_INFO]) ** 2)
    rmse = float(np.sqrt(np.mean(sq)))
    ratio = f"{rmse / prev:>8.3f}" if prev else f"{'-':>8}"
    print(f"{n:>9}{rmse:>10.4f}{ratio}")
    prev = rmse

The ratio hovers near `0.707` = $1/\sqrt{2}$: doubling the data cuts the error
by ~30%, exactly as the theory says. (With only 30 runs per row these ratios are
themselves noisy, so expect wobble around the trend rather than a clean
sequence.)

This rate is the fundamental cost of being model-free. Value iteration in
notebook 02 hit machine precision in **4 sweeps**. Monte Carlo needs thousands
of episodes for two decimal places — because it is *estimating* the expectation
that dynamic programming *computed*.

## Incremental updates

Storing running sums works, but the standard formulation is incremental. After
observing return $G$ for state $s$:

$$V(s) \leftarrow V(s) + \frac{1}{N(s)}\big[G - V(s)\big]$$

The bracketed term is an **error signal** — how surprising this return was
relative to the current estimate — and the update moves the estimate a fraction
of the way toward it. That template, `new = old + step * (target - old)`,
recurs constantly in RL; you will meet it again as the critic update in
notebook 04.

Replacing $1/N(s)$ with a constant $\alpha$ gives an exponentially-weighted
average that tracks a *changing* target instead of converging to a fixed one —
which is what you want when the policy is still improving.

In [ ]:
def mc_incremental(pi, n_episodes, rng, gamma=GAMMA, alpha=None):
    # Incremental MC. alpha=None uses the 1/N running average; a float tracks.
    V = np.zeros(N_STATES)
    count = np.zeros(N_STATES)
    act = sample_policy(pi)

    for _ in range(n_episodes):
        traj = rollout(act, rng)
        Gs = returns_along(traj, gamma)
        seen = set()
        for (s, _, _), G in zip(traj, Gs):
            if s in seen:
                continue
            seen.add(s)
            count[s] += 1
            step_size = (1.0 / count[s]) if alpha is None else alpha
            V[s] += step_size * (G - V[s])       # new = old + step * (target - old)
    return V


rng = np.random.default_rng(3)
V_run = mc_incremental(optimal_pi, 20_000, rng)
rng = np.random.default_rng(3)
V_a05 = mc_incremental(optimal_pi, 20_000, rng, alpha=0.05)

print(f"{'state':<18}{'1/N average':>13}{'alpha=0.05':>13}{'exact':>10}")
for s in NONTERMINAL:
    if counts[s] > 0:
        print(f"{s.name:<18}{V_run[s]:>13.4f}{V_a05[s]:>13.4f}{V_TRUE[s]:>10.4f}")

The `1/N` average is more accurate here, and that is expected: the policy is
fixed, so the target never moves and averaging everything equally is optimal.
The constant-$\alpha$ version keeps giving recent episodes disproportionate
weight, so it never fully settles — it jitters around the answer forever.

That apparent flaw is the reason to use it. When the policy *is* changing, old
returns describe a policy that no longer exists, and forgetting them is exactly
right.

## Evaluating a stochastic policy

Everything so far evaluated a deterministic policy. Nothing in Monte Carlo
requires that — the method only needs episodes. Below we evaluate an
$\varepsilon$-greedy version of the optimal policy, which takes a random action
$\varepsilon$ of the time.

This also fixes the coverage gap from earlier: with random exploration,
`FOULING_CHECKED` finally gets visited.

In [ ]:
def epsilon_greedy(pi, eps):
    # Mix a deterministic policy with uniform random action selection.
    return (1 - eps) * pi + eps / N_ACTIONS


eps_pi = epsilon_greedy(optimal_pi, 0.2)
V_eps_true = evaluate_exact(eps_pi)

rng = np.random.default_rng(5)
V_eps_mc, counts_eps, _ = mc_evaluate(eps_pi, 30_000, rng)

print("epsilon-greedy (eps=0.2) policy:")
print(f"{'state':<18}{'MC':>10}{'exact':>10}{'error':>10}{'visits':>9}")
for s in NONTERMINAL:
    print(f"{s.name:<18}{V_eps_mc[s]:>10.4f}{V_eps_true[s]:>10.4f}"
          f"{V_eps_mc[s] - V_eps_true[s]:>+10.4f}{int(counts_eps[s]):>9}")

print(f"\noptimal policy value at NO_INFO:       {V_TRUE[State.NO_INFO]:.4f}")
print(f"epsilon-greedy value at NO_INFO:       {V_eps_true[State.NO_INFO]:.4f}")
print(f"cost of exploring:                     {V_eps_true[State.NO_INFO] - V_TRUE[State.NO_INFO]:+.4f}")

Two results worth noting.

**Coverage is fixed.** Every state now has a healthy visit count, so every state
gets a real estimate. Exploration bought us the information that greedy
behaviour could not.

**Exploration is not free.** The $\varepsilon$-greedy policy is genuinely worse
— it sometimes takes actions it knows to be bad. That number is the price of
the information, and the explore/exploit trade-off in one line: you cannot learn
about actions you never take, and taking them costs you return.

## What Monte Carlo gives, and what it costs

**Gives:**
- no model needed — only sampled experience
- unbiased (first-visit), and converges to the true $v_\pi$
- each state's estimate is independent of the others' errors, since nothing
  bootstraps

**Costs:**
- $1/\sqrt{N}$ convergence — slow next to dynamic programming
- **episodes must terminate**; there are no updates until the end
- high variance, since full returns accumulate every random outcome along the way
- only evaluates states the policy visits

The variance issue is the one that drives the rest of the series. Monte Carlo
returns are noisy, and in notebook 04 we use exactly these returns to estimate a
**policy gradient** — inheriting all of that noise. Cutting it down is what
baselines and critics are for.

| Notebook | Adds |
| --- | --- |
| 01 | states, actions, transitions, rewards, terminal states |
| 02 | Bellman equations, value iteration, policy extraction |
| **03 — this one** | Monte Carlo policy evaluation from sampled episodes |
| **04** | REINFORCE, then baselines and a critic to cut its variance |
| **05** | PPO's probability ratio and clipping |

So far we have only *evaluated* policies given to us. Notebook 04 starts
**improving** one directly, by gradient ascent on its parameters.